# Experiment 4: Implement Basic Authentication (FastAPI)

**Objective:**
- Implement API Key and JWT-based authentication
- Secure the prediction endpoints
- Create login/token generation endpoint

**Prerequisites:** Run Experiments 1-3 first.

## Step 1: Install Required Libraries

In [ ]:
!pip install fastapi uvicorn pydantic scikit-learn pandas numpy nest-asyncio python-jose[cryptography] passlib[bcrypt] python-multipart

## Step 2: Import Libraries

In [ ]:
import pickle
import numpy as np
import pandas as pd
import uvicorn
import nest_asyncio
import threading
import time
import requests
import json
from datetime import datetime, timedelta

from fastapi import FastAPI, Request, HTTPException, Depends, Security
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials, APIKeyHeader
from pydantic import BaseModel, Field
from typing import Literal, Optional
from jose import JWTError, jwt
from passlib.context import CryptContext

nest_asyncio.apply()
print("All libraries imported successfully!")

## Step 3: Authentication Configuration

In [ ]:
# JWT Configuration
SECRET_KEY = "mlops-secret-key-2024-change-in-production"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# API Key Configuration
VALID_API_KEYS = [
    "mlops-api-key-001",
    "mlops-api-key-002",
    "test-api-key-999"
]

# Simulated user database
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

USERS_DB = {
    "admin": {
        "username": "admin",
        "hashed_password": pwd_context.hash("admin123"),
        "role": "admin"
    },
    "user1": {
        "username": "user1",
        "hashed_password": pwd_context.hash("user123"),
        "role": "user"
    }
}

# Security schemes
bearer_scheme = HTTPBearer()
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

print("Authentication configuration done!")
print(f"JWT Algorithm: {ALGORITHM}")
print(f"Token Expiry: {ACCESS_TOKEN_EXPIRE_MINUTES} minutes")
print(f"Users: {list(USERS_DB.keys())}")
print(f"API Keys: {VALID_API_KEYS}")

## Step 4: JWT Token Functions

In [ ]:
def verify_password(plain_password, hashed_password):
    """Verify a password against its hash."""
    return pwd_context.verify(plain_password, hashed_password)

def authenticate_user(username: str, password: str):
    """Authenticate a user by username and password."""
    user = USERS_DB.get(username)
    if not user:
        return None
    if not verify_password(password, user["hashed_password"]):
        return None
    return user

def create_access_token(data: dict, expires_delta: Optional[timedelta] = None):
    """Create a JWT access token."""
    to_encode = data.copy()
    expire = datetime.utcnow() + (expires_delta or timedelta(minutes=15))
    to_encode.update({"exp": expire})
    encoded_jwt = jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)
    return encoded_jwt

def verify_jwt_token(token: str):
    """Verify a JWT token and return the payload."""
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username = payload.get("sub")
        if username is None:
            raise HTTPException(status_code=401, detail="Invalid token: no subject")
        return payload
    except JWTError as e:
        raise HTTPException(status_code=401, detail=f"Invalid token: {str(e)}")

# Test token creation
test_token = create_access_token({"sub": "admin", "role": "admin"}, timedelta(minutes=30))
print(f"Sample JWT Token: {test_token[:50]}...")
print(f"Token verified: {verify_jwt_token(test_token)}")

## Step 5: Authentication Dependencies

In [ ]:
# Dependency: Verify API Key
async def verify_api_key(api_key: str = Security(api_key_header)):
    """Verify the API key from header."""
    if api_key is None:
        raise HTTPException(status_code=401, detail="API Key is missing")
    if api_key not in VALID_API_KEYS:
        raise HTTPException(status_code=403, detail="Invalid API Key")
    return api_key

# Dependency: Verify JWT Bearer Token
async def verify_bearer_token(credentials: HTTPAuthorizationCredentials = Security(bearer_scheme)):
    """Verify the JWT bearer token."""
    token = credentials.credentials
    payload = verify_jwt_token(token)
    return payload

# Combined: Accept either API Key OR JWT Token
async def authenticate(
    api_key: Optional[str] = Security(api_key_header),
    credentials: Optional[HTTPAuthorizationCredentials] = Security(HTTPBearer(auto_error=False))
):
    """Authenticate using either API Key or JWT token."""
    # Try API Key first
    if api_key and api_key in VALID_API_KEYS:
        return {"auth_method": "api_key", "key": api_key}
    
    # Try JWT token
    if credentials:
        payload = verify_jwt_token(credentials.credentials)
        return {"auth_method": "jwt", "user": payload.get("sub"), "role": payload.get("role")}
    
    raise HTTPException(
        status_code=401,
        detail="Authentication required. Provide API Key (X-API-Key header) or JWT Bearer token."
    )

print("Authentication dependencies defined!")

## Step 6: Load Model & Define Schemas

In [ ]:
# Load model artifacts
with open('model_artifacts/churn_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('model_artifacts/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('model_artifacts/label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

# Schemas
class LoginRequest(BaseModel):
    username: str = Field(..., description="Username")
    password: str = Field(..., description="Password")

class TokenResponse(BaseModel):
    access_token: str
    token_type: str = "bearer"
    expires_in: int

class ChurnPredictionRequest(BaseModel):
    Gender: Literal['Male', 'Female']
    SeniorCitizen: int = Field(..., ge=0, le=1)
    Tenure: int = Field(..., ge=0)
    MonthlyCharges: float = Field(..., ge=0)
    Contract: Literal['Month-to-month', 'One year', 'Two year']
    PaymentMethod: Literal['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']
    TotalCharges: float = Field(..., ge=0)

class ChurnPredictionResponse(BaseModel):
    prediction: int
    prediction_label: str
    churn_probability: float
    no_churn_probability: float
    authenticated_via: str

print("Model and schemas loaded!")

## Step 7: Create FastAPI App with Authentication

In [ ]:
app = FastAPI(
    title="Bank Churn Prediction API (Authenticated)",
    description="API with API Key and JWT Authentication",
    version="3.0.0"
)

# ====================== PUBLIC ENDPOINTS ======================

@app.get("/", tags=["Public"])
def root():
    return {"message": "Bank Churn Prediction API v3.0 (Authenticated)", "docs": "/docs"}

@app.get("/health", tags=["Public"])
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

# ====================== LOGIN ENDPOINT ======================

@app.post("/login", response_model=TokenResponse, tags=["Authentication"])
def login(login_data: LoginRequest):
    """Authenticate and get a JWT token."""
    user = authenticate_user(login_data.username, login_data.password)
    if not user:
        raise HTTPException(status_code=401, detail="Invalid username or password")
    
    access_token = create_access_token(
        data={"sub": user["username"], "role": user["role"]},
        expires_delta=timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    )
    
    return TokenResponse(
        access_token=access_token,
        token_type="bearer",
        expires_in=ACCESS_TOKEN_EXPIRE_MINUTES * 60
    )

# ====================== SECURED ENDPOINTS ======================

@app.post("/predict", response_model=ChurnPredictionResponse, tags=["Prediction"])
def predict_churn(
    request_data: ChurnPredictionRequest,
    auth: dict = Depends(authenticate)
):
    """Predict churn (requires authentication via API Key or JWT)."""
    input_data = pd.DataFrame([request_data.model_dump()])
    for col in label_encoders:
        if col in input_data.columns:
            input_data[col] = label_encoders[col].transform(input_data[col])
    input_scaled = scaler.transform(input_data)
    prediction = model.predict(input_scaled)[0]
    probabilities = model.predict_proba(input_scaled)[0]
    
    return ChurnPredictionResponse(
        prediction=int(prediction),
        prediction_label="Churn" if prediction == 1 else "No Churn",
        churn_probability=round(float(probabilities[1]), 4),
        no_churn_probability=round(float(probabilities[0]), 4),
        authenticated_via=auth.get("auth_method", "unknown")
    )

# API Key only endpoint
@app.post("/predict/api-key", response_model=ChurnPredictionResponse, tags=["Prediction"])
def predict_with_api_key(
    request_data: ChurnPredictionRequest,
    api_key: str = Depends(verify_api_key)
):
    """Predict churn (requires API Key only)."""
    input_data = pd.DataFrame([request_data.model_dump()])
    for col in label_encoders:
        if col in input_data.columns:
            input_data[col] = label_encoders[col].transform(input_data[col])
    input_scaled = scaler.transform(input_data)
    prediction = model.predict(input_scaled)[0]
    probabilities = model.predict_proba(input_scaled)[0]
    
    return ChurnPredictionResponse(
        prediction=int(prediction),
        prediction_label="Churn" if prediction == 1 else "No Churn",
        churn_probability=round(float(probabilities[1]), 4),
        no_churn_probability=round(float(probabilities[0]), 4),
        authenticated_via="api_key"
    )

# JWT only endpoint
@app.post("/predict/jwt", response_model=ChurnPredictionResponse, tags=["Prediction"])
def predict_with_jwt(
    request_data: ChurnPredictionRequest,
    token_data: dict = Depends(verify_bearer_token)
):
    """Predict churn (requires JWT token only)."""
    input_data = pd.DataFrame([request_data.model_dump()])
    for col in label_encoders:
        if col in input_data.columns:
            input_data[col] = label_encoders[col].transform(input_data[col])
    input_scaled = scaler.transform(input_data)
    prediction = model.predict(input_scaled)[0]
    probabilities = model.predict_proba(input_scaled)[0]
    
    return ChurnPredictionResponse(
        prediction=int(prediction),
        prediction_label="Churn" if prediction == 1 else "No Churn",
        churn_probability=round(float(probabilities[1]), 4),
        no_churn_probability=round(float(probabilities[0]), 4),
        authenticated_via="jwt"
    )

print("FastAPI app with authentication created!")
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f"  {list(route.methods)} {route.path}")

## Step 8: Write Authenticated App to File

In [ ]:
app_code = '''import pickle
import numpy as np
import pandas as pd
import uvicorn
import logging
import json
import uuid
import time
import traceback
import os
from datetime import datetime, timedelta

from fastapi import FastAPI, Request, HTTPException, Depends, Security
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials, APIKeyHeader
from pydantic import BaseModel, Field
from typing import Literal, Optional
from starlette.middleware.base import BaseHTTPMiddleware
from jose import JWTError, jwt
from passlib.context import CryptContext

# ====================== CONFIG ======================
SECRET_KEY = os.getenv("SECRET_KEY", "mlops-secret-key-2024-change-in-production")
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30
VALID_API_KEYS = os.getenv("API_KEYS", "mlops-api-key-001,mlops-api-key-002").split(",")

# ====================== AUTH SETUP ======================
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
bearer_scheme = HTTPBearer(auto_error=False)
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

USERS_DB = {
    "admin": {"username": "admin", "hashed_password": pwd_context.hash("admin123"), "role": "admin"},
    "user1": {"username": "user1", "hashed_password": pwd_context.hash("user123"), "role": "user"},
}

def verify_password(plain, hashed):
    return pwd_context.verify(plain, hashed)

def authenticate_user(username, password):
    user = USERS_DB.get(username)
    if not user or not verify_password(password, user["hashed_password"]):
        return None
    return user

def create_access_token(data: dict, expires_delta=None):
    to_encode = data.copy()
    to_encode.update({"exp": datetime.utcnow() + (expires_delta or timedelta(minutes=15))})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def verify_jwt_token(token: str):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        if not payload.get("sub"):
            raise HTTPException(status_code=401, detail="Invalid token")
        return payload
    except JWTError as e:
        raise HTTPException(status_code=401, detail=f"Invalid token: {str(e)}")

async def authenticate(api_key: Optional[str] = Security(api_key_header), credentials: Optional[HTTPAuthorizationCredentials] = Security(HTTPBearer(auto_error=False))):
    if api_key and api_key in VALID_API_KEYS:
        return {"auth_method": "api_key", "key": api_key}
    if credentials:
        payload = verify_jwt_token(credentials.credentials)
        return {"auth_method": "jwt", "user": payload.get("sub"), "role": payload.get("role")}
    raise HTTPException(status_code=401, detail="Authentication required")

# ====================== LOAD MODEL ======================
with open("model_artifacts/churn_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("model_artifacts/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
with open("model_artifacts/label_encoders.pkl", "rb") as f:
    label_encoders = pickle.load(f)

# ====================== SCHEMAS ======================
class LoginRequest(BaseModel):
    username: str
    password: str

class TokenResponse(BaseModel):
    access_token: str
    token_type: str = "bearer"
    expires_in: int

class ChurnPredictionRequest(BaseModel):
    Gender: Literal["Male", "Female"]
    SeniorCitizen: int = Field(..., ge=0, le=1)
    Tenure: int = Field(..., ge=0)
    MonthlyCharges: float = Field(..., ge=0)
    Contract: Literal["Month-to-month", "One year", "Two year"]
    PaymentMethod: Literal["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"]
    TotalCharges: float = Field(..., ge=0)

class ChurnPredictionResponse(BaseModel):
    prediction: int
    prediction_label: str
    churn_probability: float
    no_churn_probability: float
    authenticated_via: str

# ====================== APP ======================
app = FastAPI(title="Bank Churn Prediction API", version="3.0.0")

@app.get("/")
def root():
    return {"message": "Bank Churn Prediction API v3.0"}

@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": model is not None}

@app.post("/login", response_model=TokenResponse)
def login(data: LoginRequest):
    user = authenticate_user(data.username, data.password)
    if not user:
        raise HTTPException(status_code=401, detail="Invalid credentials")
    token = create_access_token({"sub": user["username"], "role": user["role"]}, timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES))
    return TokenResponse(access_token=token, expires_in=ACCESS_TOKEN_EXPIRE_MINUTES * 60)

@app.post("/predict", response_model=ChurnPredictionResponse)
def predict(request_data: ChurnPredictionRequest, auth: dict = Depends(authenticate)):
    input_data = pd.DataFrame([request_data.model_dump()])
    for col in label_encoders:
        if col in input_data.columns:
            input_data[col] = label_encoders[col].transform(input_data[col])
    input_scaled = scaler.transform(input_data)
    prediction = model.predict(input_scaled)[0]
    probs = model.predict_proba(input_scaled)[0]
    return ChurnPredictionResponse(
        prediction=int(prediction),
        prediction_label="Churn" if prediction == 1 else "No Churn",
        churn_probability=round(float(probs[1]), 4),
        no_churn_probability=round(float(probs[0]), 4),
        authenticated_via=auth.get("auth_method", "unknown")
    )

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("Authenticated app.py written!")

## Step 9: Run and Test Authentication

In [ ]:
# Start server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("Server started on http://localhost:8000")

In [ ]:
test_payload = {
    "Gender": "Male", "SeniorCitizen": 0, "Tenure": 30,
    "MonthlyCharges": 70.5, "Contract": "One year",
    "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
}

# Test 1: No authentication - should fail
print("=" * 50)
print("TEST 1: No Authentication (should fail)")
print("=" * 50)
response = requests.post("http://localhost:8000/predict", json=test_payload)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

In [ ]:
# Test 2: API Key authentication
print("=" * 50)
print("TEST 2: API Key Authentication")
print("=" * 50)
response = requests.post(
    "http://localhost:8000/predict",
    json=test_payload,
    headers={"X-API-Key": "mlops-api-key-001"}
)
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

In [ ]:
# Test 3: Invalid API Key
print("=" * 50)
print("TEST 3: Invalid API Key (should fail)")
print("=" * 50)
response = requests.post(
    "http://localhost:8000/predict",
    json=test_payload,
    headers={"X-API-Key": "wrong-key"}
)
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

In [ ]:
# Test 4: JWT Login
print("=" * 50)
print("TEST 4: JWT Login")
print("=" * 50)
login_response = requests.post("http://localhost:8000/login", json={
    "username": "admin",
    "password": "admin123"
})
print(f"Login Status: {login_response.status_code}")
token_data = login_response.json()
print(f"Token: {token_data['access_token'][:50]}...")
print(f"Expires in: {token_data['expires_in']} seconds")

In [ ]:
# Test 5: JWT Token authentication
print("=" * 50)
print("TEST 5: JWT Token Authentication")
print("=" * 50)
jwt_token = token_data['access_token']
response = requests.post(
    "http://localhost:8000/predict",
    json=test_payload,
    headers={"Authorization": f"Bearer {jwt_token}"}
)
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

In [ ]:
# Test 6: Invalid credentials
print("=" * 50)
print("TEST 6: Invalid Login Credentials")
print("=" * 50)
response = requests.post("http://localhost:8000/login", json={
    "username": "admin",
    "password": "wrongpassword"
})
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

print("\n✅ Authentication tests completed!")